# Visualizacao: sifilis congenita em Porto Alegre

Este notebook cria uma visualizacao simples, mas alinhada ao anteprojeto sobre desigualdade racial na sifilis congenita em Porto Alegre.

A ideia central e comparar a composicao racial das maes em duas bases de 2024:

- SINASC: nascidos vivos em Porto Alegre, usado como contexto/denominador.
- SINAN/SIFCBR: notificacoes de sifilis congenita em residentes de Porto Alegre, usado como desfecho.

A visualizacao final pergunta: a participacao de maes negras entre os casos notificados de sifilis congenita e maior do que entre os nascidos vivos do municipio?


## 1. Dependencias

No Colab, execute esta celula uma vez. O extra `dbc` do PySUS e usado porque os arquivos brutos do DATASUS estao em formato `.dbc`.


In [ ]:
!pip -q install "pysus[dbc]==2.0.4" pandas matplotlib seaborn


## 2. Bibliotecas e configuracoes


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CODIGO_PORTO_ALEGRE = "431490"


## 3. Caminhos dos dados

A estrutura esperada no repositorio e:

- `data/raw/SIFCBR24.dbc`
- `data/raw/sinasc/DNRS2024.dbc`
- `data/raw/cnes/STRS24*.dbc`


In [ ]:
def encontrar_arquivo(*candidatos):
    for candidato in candidatos:
        caminho = Path(candidato)
        if caminho.exists():
            return caminho
    caminhos = ", ".join(str(c) for c in candidatos)
    raise FileNotFoundError(f"Nenhum arquivo encontrado. Caminhos testados: {caminhos}")

ARQUIVO_SINAN = encontrar_arquivo(
    "../data/raw/SIFCBR24.dbc",
    "data/raw/SIFCBR24.dbc",
    "SIFCBR24.dbc",
)

ARQUIVO_SINASC = encontrar_arquivo(
    "../data/raw/sinasc/DNRS2024.dbc",
    "data/raw/sinasc/DNRS2024.dbc",
    "DNRS2024.dbc",
)

print("SINAN:", ARQUIVO_SINAN)
print("SINASC:", ARQUIVO_SINASC)


## 4. Leitura dos arquivos `.dbc`

O PySUS converte os arquivos comprimidos do DATASUS para tabelas que podem ser manipuladas com pandas.


In [ ]:
try:
    from pysus.utilities.readdbc import read_dbc
except ImportError as exc:
    raise ImportError(
        "Nao foi possivel importar read_dbc. Verifique a instalacao de pysus[dbc]."
    ) from exc

sinan = read_dbc(str(ARQUIVO_SINAN), encoding="latin1")
sinasc = read_dbc(str(ARQUIVO_SINASC), encoding="latin1")

print("SINAN", sinan.shape)
print("SINASC", sinasc.shape)


## 5. Recorte: Porto Alegre

Nos microdados, o codigo de Porto Alegre aparece como `431490` ou com digito adicional. Por isso o filtro usa `startswith("431490")`.


In [ ]:
sinan_poa = sinan[
    sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

sinasc_poa = sinasc[
    sinasc["CODMUNRES"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

print("Casos SINAN Porto Alegre:", len(sinan_poa))
print("Nascidos vivos SINASC Porto Alegre:", len(sinasc_poa))


## 6. Preparacao das variaveis

Para dialogar com o anteprojeto, a analise separa maes negras de maes nao negras. No padrao do SUS, populacao negra combina as categorias preta e parda.


In [ ]:
def grupo_raca(valor):
    codigo = str(valor).strip()
    if codigo in {"2", "4"}:
        return "Maes negras"
    if codigo in {"1", "3", "5"}:
        return "Maes nao negras"
    return "Ignorado/sem informacao"

sinan_poa["grupo_raca"] = sinan_poa["ANT_RACA"].map(grupo_raca)
sinasc_poa["grupo_raca"] = sinasc_poa["RACACORMAE"].map(grupo_raca)

base_comparacao = pd.concat(
    [
        sinasc_poa.assign(base="Nascidos vivos (SINASC)")[["base", "grupo_raca"]],
        sinan_poa.assign(base="Casos de sifilis congenita (SINAN)")[["base", "grupo_raca"]],
    ],
    ignore_index=True,
)

base_comparacao = base_comparacao[
    base_comparacao["grupo_raca"].isin(["Maes negras", "Maes nao negras"])
]

resumo_raca = (
    base_comparacao
    .groupby(["base", "grupo_raca"], as_index=False)
    .size()
    .rename(columns={"size": "registros"})
)

resumo_raca["percentual"] = resumo_raca.groupby("base")["registros"].transform(
    lambda valores: valores / valores.sum() * 100
)

resumo_raca


## 7. Indicador auxiliar: pre-natal nos casos notificados

Este painel ajuda a manter o grafico conectado ao argumento do anteprojeto: sifilis congenita e um evento evitavel e relacionado a acesso/qualidade do pre-natal.


In [ ]:
mapa_prenatal = {
    "1": "Realizou pre-natal",
    "2": "Nao realizou pre-natal",
    "9": "Ignorado/sem informacao",
}

prenatal = sinan_poa.copy()
prenatal["grupo_prenatal"] = prenatal["ANT_PRE_NA"].astype(str).str.strip().map(mapa_prenatal)
prenatal["grupo_prenatal"] = prenatal["grupo_prenatal"].fillna("Ignorado/sem informacao")

resumo_prenatal = (
    prenatal
    .groupby("grupo_prenatal", as_index=False)
    .size()
    .rename(columns={"size": "casos"})
    .sort_values("casos", ascending=False)
)

resumo_prenatal["percentual"] = resumo_prenatal["casos"] / resumo_prenatal["casos"].sum() * 100
resumo_prenatal


## 8. Visualizacao final

A figura combina dois paineis: composicao racial comparada entre SINASC e SINAN, e situacao de pre-natal entre os casos de sifilis congenita.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.25, 1]})

ordem_bases = ["Nascidos vivos (SINASC)", "Casos de sifilis congenita (SINAN)"]
paleta_raca = {
    "Maes negras": "#B55A30",
    "Maes nao negras": "#4C78A8",
}

sns.barplot(
    data=resumo_raca,
    x="percentual",
    y="base",
    hue="grupo_raca",
    order=ordem_bases,
    palette=paleta_raca,
    ax=axes[0],
)

axes[0].set_title("Composicao racial: nascidos vivos vs. casos notificados", weight="bold")
axes[0].set_xlabel("Percentual dentro de cada base")
axes[0].set_ylabel("")
axes[0].set_xlim(0, 100)
axes[0].legend(title="Grupo racial", loc="lower right")

for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.1f%%", padding=3, fontsize=9)

plot_prenatal = resumo_prenatal.sort_values("percentual", ascending=True)
cores_prenatal = ["#72B7B2" if "Realizou" in item else "#E45756" for item in plot_prenatal["grupo_prenatal"]]

axes[1].barh(plot_prenatal["grupo_prenatal"], plot_prenatal["percentual"], color=cores_prenatal)
axes[1].set_title("Pre-natal entre casos notificados", weight="bold")
axes[1].set_xlabel("Percentual dos casos")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 100)

for i, row in enumerate(plot_prenatal.itertuples(index=False)):
    axes[1].text(row.percentual + 1, i, f"{row.percentual:.1f}%", va="center", fontsize=9)

for ax in axes:
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)

fig.suptitle(
    "Sifilis congenita em Porto Alegre (2024): desigualdade racial e acesso ao pre-natal",
    fontsize=14,
    weight="bold",
    y=1.03,
)

plt.tight_layout()

saida_dir = Path("../outputs/figures")
if not saida_dir.exists():
    saida_dir = Path("outputs/figures")
saida_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(saida_dir / "visualizacao_sifilis_congenita_poars.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Como usar no relatorio

Imagem gerada: `outputs/figures/visualizacao_sifilis_congenita_poars.png`.

Caption sugerido para voce adaptar com suas palavras: a figura compara a composicao racial das maes de nascidos vivos em Porto Alegre com a composicao racial das maes associadas a notificacoes de sifilis congenita no mesmo municipio e ano. O painel complementar mostra a distribuicao dos casos segundo realizacao de pre-natal, conectando o desfecho a um marcador de acesso ao cuidado.
